In [15]:
import os
import time
import numpy as np
from PIL import Image
from transformers import pipeline

def infer_with_pipeline(image_path, model_name="depth-anything/Depth-Anything-V2-Small-hf"):
    depth_pipe = pipeline(
        task="depth-estimation",
        model=model_name
    )
    print(f"모델: {model_name}")

    result = depth_pipe(image_path)

    depth_image = result["depth"]
    depth_array = np.array(depth_image)

    print(f"입력: {image_path}")
    print(f"출력 크기: {depth_array.shape}")
    print(f"값 범위: [{depth_array.min()}, {depth_array.max()}]")
    print(f"데이터 타입: {depth_array.dtype}")

    os.makedirs("outputs", exist_ok=True)
    depth_image.save("outputs/depth_pipeline_result.png")
    print(f"저장: outputs/depth_pipeline_result.png")

    return depth_array

def compare_models(image_path):
    models = [
        ("depth-anything/Depth-Anything-V2-Small-hf", "ViT-S"),
        ("depth-anything/Depth-Anything-V2-Base-hf", "ViT-B"),
    ]

    for model_name, desc in models:
        pipe = pipeline("depth-estimation", model=model_name)

        pipe(image_path)
        times = []
        for _ in range(5):
            start = time.time()
            result = pipe(image_path)
            times.append(time.time() - start)

        avg_ms = np.mean(times) * 1000
        print(f"{desc}: {avg_ms:.1f} ms/frame ({1000/avg_ms:.1f} FPS)")

if __name__ == "__main__":
    image_path = "data/indoor.jpg"
    depth = infer_with_pipeline(image_path)

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 17984.09it/s]


모델: depth-anything/Depth-Anything-V2-Small-hf
입력: data/indoor.jpg
출력 크기: (1200, 1607)
값 범위: [0, 255]
데이터 타입: uint8
저장: outputs/depth_pipeline_result.png


In [16]:
import torch
import numpy as np
import cv2
import time
from transformers import AutoModelForDepthEstimation
from transformers import AutoImageProcessor
from PIL import Image

class DepthAnythingInference:
    def __init__(self, model_name="depth-anything/Depth-Anything-V2-Small-hf", device=None):
        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)

        print(f"장치: {self.device}")

        self.model = AutoModelForDepthEstimation.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

        self.processor = AutoImageProcessor.from_pretrained(model_name)

        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"파라미터: {total_params / 1e6:.1f}M")

    def infer(self, image):
        if isinstance(image, np.ndarray):
            image_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        else:
            image_pil = image

        orig_size = image_pil.size

        inputs = self.processor(images=image_pil, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        depth = outputs.predicted_depth.squeeze()
        depth = torch.nn.functional.interpolate(
            depth.unsqueeze(0).unsqueeze(0),
            size=(orig_size[1], orig_size[0]),
            mode="bicubic",
            align_corners=False            
        ).squeeze()

        depth_numpy = depth.cpu().numpy()

        return depth_numpy

    def infer_batch(self, images):
        inputs = self.processor(images=images, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        depths = outputs.predicted_depth

        results = []
        for i, img in enumerate(images):
            d = depths[i].unsqueeze(0).unsqueeze(0)
            d = torch.nn.functional.interpolate(
                d, size=img.size[::-1], mode="bicubic", align_corners=False
            ).squeeze()
            results.append(d.cpu().numpy())

        return results

    def benchmark(self, image, num_runs=20):
        for _ in range(3):
            self.infer(image)

        times = []
        for _ in range(num_runs):
            start = time.time()
            self.infer(image)
            times.append(time.time() - start)

        avg_ms = np.mean(times) * 1000
        std_ms = np.std(times) * 1000
        fps = 1000 / avg_ms

        print(f"\n 벤치마크 결과 ({num_runs}회):")
        print(f"평균: {avg_ms:.1f} ms (+/- {std_ms:.1f})")
        print(f"FPS: {fps:.1f}")


        return avg_ms

if __name__ == "__main__":
    inferencer = DepthAnythingInference() 

    image = Image.open("data/indoor.jpg")
    print(f"이미지 크기: {image.size}")


    depth_map = inferencer.infer(image) 
    print(f"깊이맵 크기: {depth_map.shape}")
    print(f"깊이 범위: [{depth_map.min():.3f}, {depth_map.max():.3f}]")

    depth_maps = inferencer.infer_batch([image, image]) 
    print(f"배치 결과: {len(depth_maps)}장, 각 shape={depth_maps[0].shape}")

    inferencer.benchmark(image)

장치: cuda


Loading weights: 100%|██████████| 287/287 [00:00<00:00, 18467.47it/s]


파라미터: 24.8M
이미지 크기: (1607, 1200)
깊이맵 크기: (1200, 1607)
깊이 범위: [0.138, 6.252]
배치 결과: 2장, 각 shape=(1200, 1607)

 벤치마크 결과 (20회):
평균: 25.9 ms (+/- 0.5)
FPS: 38.7


In [17]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline


def normalize_depth(depth_map):
    d_min = depth_map.min()
    d_max = depth_map.max()
    if d_max - d_min < 1e-6:
        return np.zeros_like(depth_map)
    return (depth_map - d_min) / (d_max - d_min)

def depth_to_colormap(depth_map, colormap_name="magma"):
    depth_norm = normalize_depth(depth_map)
    cmap = plt.get_cmap(colormap_name)
    colored = cmap(depth_norm)[:, :, :3]
    colored = (colored * 255).astype(np.uint8)
    return colored

def visualize_side_by_side(image_path, depth_map, save_path="outputs/depth_comparison.png"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    image = Image.open(image_path)
    axes[0].imshow(image)
    axes[0].set_title("Original Image", fontsize=14)
    axes[0].axis("off")

    im = axes[1].imshow(depth_map, cmap="magma")
    axes[1].set_title("Depth Map (Depth Anything)", fontsize=14)
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"저장: {save_path}")
    plt.close()

def visualize_colormaps(depth_map, save_path="outputs/depth_colormaps.png"):
    colormaps = ["magma", "inferno", "turbo", "viridis", "plasma", "gray"]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for ax, cmap_name in zip(axes, colormaps):
        ax.imshow(depth_map, cmap=cmap_name)
        ax.set_title(f"Colormap: {cmap_name}", fontsize=12)
        ax.axis("off")

    plt.suptitle("Depth Colormap Comparison", fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"저장: {save_path}")
    plt.close()

def visualize_depth_histogram(depth_map, save_path="outputs/depth_histogram.png"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].imshow(depth_map, cmap="magma")
    axes[0].set_title("Depth Map")
    axes[0].axis("off")

    axes[1].hist(depth_map.flatten(), bins=100, color="steelblue", alpha=0.7)
    axes[1].set_xlabel("Depth Value")
    axes[1].set_ylabel("Pixel Count")
    axes[1].set_title("Depth Value Distribution")
    axes[1].axvline(depth_map.mean(), color="red", linestyle="--", label=f"Mean {depth_map.mean():.2f}")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"저장: {save_path}")
    plt.close()

def create_overlay(image_path, depth_map, alpha=0.5, save_path="outputs/depth_overlay.png"):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    depth_colored = depth_to_colormap(depth_map, "turbo")

    if depth_colored.shape[:2] != image.shape[:2]:
        depth_colored = cv2.resize(depth_colored, image.shape[1], image.shape[0])

    overlay = cv2.addWeighted(image, 1 - alpha, depth_colored, alpha, 0)

    plt.figure(figsize=(10, 6))
    plt.imshow(overlay)
    plt.title(f"Depth Overlay (alpha={alpha})")
    plt.axis("off")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"저장: {save_path}")
    plt.close()

if __name__ == "__main__":
    image_path = "data/indoor.jpg"

    pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")
    result = pipe(image_path)
    depth_map = np.array(result["depth"]).astype(np.float32)
    print(f"깊이맵 크기: {depth_map.shape}")

    os.makedirs("outputs", exist_ok=True)
    visualize_side_by_side(image_path, depth_map)
    visualize_colormaps(depth_map)
    visualize_depth_histogram(depth_map)
    create_overlay(image_path, depth_map, alpha=0.4)

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 13695.34it/s]


깊이맵 크기: (1200, 1607)
저장: outputs/depth_comparison.png
저장: outputs/depth_colormaps.png
저장: outputs/depth_histogram.png
저장: outputs/depth_overlay.png


In [1]:
import os
import numpy as np
import rerun as rr
from PIL import Image
from transformers import pipeline

def depth_to_pointcloud(depth_map, image_rgb, fx, fy, cx, cy):
    H, W = depth_map.shape
    xs, ys = np.meshgrid(np.arange(W), np.arange(H))

    Z = depth_map.astype(np.float32)
    X = (xs - cx) * Z / fx
    Y = (ys - cy) * Z / fy

    points = np.stack([X, Y, Z], axis=-1).reshape(-1, 3)
    colors = image_rgb.reshape(-1, 3)
    return points, colors

def main():
    image_path = "data/indoor.jpg"

    pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")
    result = pipe(image_path)
    depth_map = np.array(result["depth"]).astype(np.float32)
    image_pil = Image.open(image_path).convert("RGB").resize(
        (depth_map.shape[1], depth_map.shape[0])
    )
    image_rgb = np.array(image_pil)

    H, W = depth_map.shape
    fx = fy = float(W)
    cx, cy = W / 2.0, H / 2.0

    points, colors = depth_to_pointcloud(depth_map, image_rgb, fx, fy, cx, cy)
    print(f"point count: {points.shape[0]}")

    rr.init("depth_anything", spawn=False)

    rr.log("image/rgb", rr.Image(image_rgb))
    rr.log("image/depth", rr.DepthImage(depth_map))

    rr.log(
        "world/cam", 
        rr.Pinhole(image_from_camera=np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]]), width=W, height=H)
    )
    rr.log(
        "world/points",
        rr.Points3D(points, colors=colors, radii=0.005)
    )

    os.makedirs("outputs", exist_ok=True)
    rr.save("outputs/depth_pointcloud.rrd")
    print("저장: outputs/depth_pointcloud.rrd")
    print("로컬에서 확인: rerun outputs/depth_pointcloud.rrd")


if __name__ == "__main__":
    main()

/workspace/study/physical-ai-study/Studies/Phase 3/week8/.venv-week8/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 287/287 [00:00<00:00, 20673.31it/s]


point count: 1928400
저장: outputs/depth_pointcloud.rrd
로컬에서 확인: rerun outputs/depth_pointcloud.rrd


In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline

def relative_to_metric(depth_map, ref_points):
    if len(ref_points) < 2:
        print("경고: 최소 2개의 참조점 필요")
        return None

    rel_values = []
    metric_values = []

    for y, x, real_depth in ref_points:
        region = depth_map[max(0, y-2):y+3, max(0, x-2):x+3]
        rel_values.append(region.mean())
        metric_values.append(real_depth)

    rel_values = np.array(rel_values)
    metric_values = np.array(metric_values)

    A = np.column_stack([rel_values, np.ones_like(rel_values)])
    result = np.linalg.lstsq(A, metric_values, rcond=None)
    alpha, beta = result[0]

    print(f"Scale (alpha): {alpha:.4f}")
    print(f"Shift (beta): {beta:4f}")

    metric_depth = alpha * depth_map + beta
    metric_depth = np.clip(metric_depth, 0.0, None)

    return metric_depth

def analyze_depth_regions(depth_map, image_path):
    h, w = depth_map.shape

    top = depth_map[:h//3, :]
    mid = depth_map[h//3:2*h//3, :]
    bot = depth_map[2*h//3:, :]

    print(f"상단 (하늘/천장): 평균={top.mean():.3f}, 범위=[{top.min():.3f}, {top.max():.3f}]")
    print(f"중단 (물체): 평균={mid.mean():.3f}, 범위=[{mid.min():.3f}, {mid.max():.3f}]")
    print(f"하단 (바닥): 평균={bot.mean():.3f}, 범위=[{bot.min():.3f}, {bot.max():.3f}]")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    image = Image.open(image_path)
    axes[0, 0].imshow(image)
    axes[0, 0].set_title("Original Image")

    axes[0, 1].imshow(depth_map, cmap="magma")
    axes[0, 1].set_title("Depth Map")
    axes[0, 1].axhline(h//3, color="cyan", linewidth=1, linestyle="--")
    axes[0, 1].axhline(2*h//3, color="cyan", linewidth=1, linestyle="--")

    axes[1, 0].hist(top.flatten(), bins=50, alpha=0.5, label="Top", color="skyblue")
    axes[1, 0].hist(mid.flatten(), bins=50, alpha=0.5, label="Middle", color="orange")
    axes[1, 0].hist(bot.flatten(), bins=50, alpha=0.5, label="Bottom", color="green")
    axes[1, 0].legend()
    axes[1, 0].set_title("Depth Distribution by Region")

    center_line = depth_map[h//2, :]
    axes[1, 1].plot(center_line)
    axes[1, 1].set_title("Center Horizontal Depth Profile")
    axes[1, 1].set_xlabel("X Coordinate")
    axes[1, 1].set_ylabel("Depth Value")

    for ax in axes.flatten():
        ax.axis("off") if hasattr(ax, "images") and ax.images else None

    plt.tight_layout()
    os.makedirs("outputs", exist_ok=True)
    plt.savefig("outputs/depth_analysis.png", dpi=150, bbox_inches="tight")
    print(f"저장: outputs/depth_analysis.png")
    plt.close()

if __name__ == "__main__":
    image_path = "data/indoor.jpg"
    pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")
    result = pipe(image_path)
    depth_map = np.array(result["depth"]).astype(np.float32)

    analyze_depth_regions(depth_map, image_path)

    print("\n Metric 변환 예시:")
    print("(실제 사용 시 실제 거리를 아는 참조점 2개 이상 필요)")

    ref_points = [
        (240, 320, 2.0),
        (100, 320, 5.0),
    ]

    metric_depth = relative_to_metric(depth_map, ref_points)
    if metric_depth is not None:
        print(f"Metric 깊이 범위: [{metric_depth.min():.2f}m, {metric_depth.max():.2f}]")

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 21110.54it/s]


상단 (하늘/천장): 평균=28.235, 범위=[1.000, 133.000]
중단 (물체): 평균=72.944, 범위=[0.000, 209.000]
하단 (바닥): 평균=172.935, 범위=[49.000, 255.000]
저장: outputs/depth_analysis.png

 Metric 변환 예시:
(실제 사용 시 실제 거리를 아는 참조점 2개 이상 필요)
Scale (alpha): 0.8152
Shift (beta): -11.858695
Metric 깊이 범위: [0.00m, 196.02]


In [ ]:
import os
import numpy as np
import cv2
from PIL import Image
from transformers import pipeline
from ultralytics import YOLO

def combine_yolo_depth(image_path, yolo_boxes, depth_map):
    image = cv2.imread(image_path)
    h_img, w_img = image.shape[:2]
    h_dep, w_dep = depth_map.shape

    if (h_dep, w_dep) != (h_img, w_img):
        depth_resized = cv2.resize(depth_map, (w_img, h_img))
    else:
        depth_resized = depth_map

    print("\n 검출 결과:")
    for box in yolo_boxes:
        x1, y1, x2, y2, cls_name, conf = box

        roi_depth = depth_resized[int(y1):int(y2), int(x1):int(x2)]

        if roi_depth.size == 0:
            continue

        mean_depth = roi_depth.mean()
        min_depth = roi_depth.min()
        max_depth = roi_depth.max()

        if mean_depth > 200:
            distance_desc = "매우 가까움"
        elif mean_depth > 150:
            distance_desc = "가까움"
        elif mean_depth > 100:
            distance_desc = "중간"
        else:
            distance_desc = "멀리"

        print(f"{cls_name} (conf={conf:.2f}):")
        print(f"bbox: ({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})")
        print(f"깊이: 평균={mean_depth:.1f}, "
              f"범위=[{min_depth:.1f}, {max_depth:.1f}]")
        print(f"거리: {distance_desc}")

        color = (0, 255, 0) if mean_depth < 150 else (0, 0, 255)
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        label = f"{cls_name}: depth={mean_depth:.0f}"
        cv2.putText(image, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    os.makedirs("outputs", exist_ok=True)
    cv2.imwrite("outputs/yolo_depth_result.jpg", image)
    print(f"\n 저장: outputs/yolo_depth_result.jpg")

if __name__ == "__main__":
    image_path = "data/indoor.jpg"

    # 1. 실제 YOLO 검출: yolo11n 사전학습 모델로 객체 박스 추출
    yolo_model = YOLO("yolo11n.pt")  # nano 모델 (없으면 자동 다운로드)
    detections = yolo_model(image_path)[0]  # 단일 이미지이므로 첫 결과만 사용

    # 2. combine_yolo_depth가 기대하는 [x1, y1, x2, y2, 클래스명, conf] 형식으로 변환
    yolo_boxes = []
    for box in detections.boxes:  # 검출된 박스마다 순회
        x1, y1, x2, y2 = box.xyxy[0].tolist()  # 픽셀 좌표 (좌상단, 우하단)
        cls_name = yolo_model.names[int(box.cls[0])]  # 클래스 인덱스 -> 이름
        conf = float(box.conf[0])  # 검출 신뢰도
        yolo_boxes.append([x1, y1, x2, y2, cls_name, conf])

    print(f"YOLO 검출: {len(yolo_boxes)}개 객체")

    # 3. Depth Anything으로 깊이맵 추론
    depth_pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")
    result = depth_pipe(image_path)
    depth_map = np.array(result["depth"]).astype(np.float32)

    # 4. 박스별 깊이 분석 + 시각화
    combine_yolo_depth(image_path, yolo_boxes, depth_map)
